In [3]:
!pip install plotly


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
!pip install nbformat --upgrade


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import pandas as pd
from datetime import datetime as dt, timedelta
import plotly.express as px
import plotly.graph_objects as go
import plotly.colors

In [6]:
data = pd.read_csv('/Users/Abhijay/Downloads/archive/online_retail.csv')
data.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [7]:
data.tail()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
1067366,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680.0,France
1067367,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680.0,France
1067368,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680.0,France
1067369,581587,22138,BAKING SET 9 PIECE RETROSPOT,3,2011-12-09 12:50:00,4.95,12680.0,France
1067370,581587,POST,POSTAGE,1,2011-12-09 12:50:00,18.00,12680.0,France


In [8]:
data.dropna(subset=['Customer ID'], inplace=True)

In [9]:
data['InvoiceDate'] = pd.to_datetime(data['InvoiceDate'])
data['Total Amount'] = data['Quantity'] * data['Price']

In [10]:
data.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Total Amount
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,30.0


In [11]:
reference_date = pd.Timestamp(dt.now().date())

In [12]:
reference_date

Timestamp('2026-10-15 00:00:00')

In [13]:
reference_date = data['InvoiceDate'].max() + timedelta(days=1)

In [14]:
reference_date

Timestamp('2011-12-10 12:50:00')

In [15]:
rfm = data.groupby('Customer ID').agg({
    'InvoiceDate': lambda x: (reference_date - x.max()).days,
    'Invoice': 'count',
    'Total Amount':'sum'
})

In [16]:
rfm.rename(columns={'InvoiceDate': 'Recency', 'Invoice': 'Frequency', 'Total Amount': 'Value'}, inplace=True)

In [17]:
rfm

,Recency,Frequency,Value
Customer ID,,,
12346.0,326,48,-64.68
12347.0,2,253,5633.32
12348.0,75,51,2019.40
12349.0,19,180,4404.54
12350.0,310,17,334.40
...,...,...,...
18283.0,4,986,2736.65
18284.0,430,29,436.68
18285.0,661,12,427.00


In [18]:
rfm.head()

,Recency,Frequency,Value
Customer ID,,,
12346.0,326,48,-64.68
12347.0,2,253,5633.32
12348.0,75,51,2019.40
12349.0,19,180,4404.54
12350.0,310,17,334.40


In [19]:
#Define quantiles
quantiles = rfm.quantile(q=[0.25, 0.5, 0.75])

#Assign RFM scores
def RScore(x,p,d):
    if p == 'Recency':
        if x <= d[p][0.25]:
            return 4
        elif x <= d[p][0.50]:
            return 3
        elif x <= d[p][0.75]:
            return 2
        else:
            return 1
    else:
        if x <= d[p][0.25]:
            return 1
        elif x <= d[p][0.50]:
            return 2
        elif x <= d[p][0.75]:
            return 3
        else:
            return 4

rfm['R'] = rfm['Recency'].apply(RScore, args=('Recency', quantiles))
rfm['F'] = rfm['Frequency'].apply(RScore, args=('Frequency', quantiles))
rfm['M'] = rfm['Value'].apply(RScore, args=('Value', quantiles))

rfm

,Recency,Frequency,Value,R,F,M
Customer ID,,,,,,
12346.0,326,48,-64.68,2,2,1
12347.0,2,253,5633.32,4,4,4
12348.0,75,51,2019.40,3,2,3
12349.0,19,180,4404.54,4,4,4
12350.0,310,17,334.40,2,1,2
...,...,...,...,...,...,...
18283.0,4,986,2736.65,4,4,4
18284.0,430,29,436.68,1,2,2
18285.0,661,12,427.00,1,1,2


In [20]:
rfm['RFM_Segment'] = rfm['R'].astype(str) + rfm['F'].astype(str) + rfm['M'].astype(str)
rfm['RFM_Score'] = rfm[['R', 'F', 'M']].sum(axis=1)



In [22]:
rfm

,Recency,Frequency,Value,R,F,M,RFM_Segment,RFM_Score
Customer ID,,,,,,,,
12346.0,326,48,-64.68,2,2,1,221,5
12347.0,2,253,5633.32,4,4,4,444,12
12348.0,75,51,2019.40,3,2,3,323,8
12349.0,19,180,4404.54,4,4,4,444,12
12350.0,310,17,334.40,2,1,2,212,5
...,...,...,...,...,...,...,...,...
18283.0,4,986,2736.65,4,4,4,444,12
18284.0,430,29,436.68,1,2,2,122,5
18285.0,661,12,427.00,1,1,2,112,4


In [24]:
segment_lables = ['Low-Value', 'Mid-Value', 'High-Value']

def assign_segment(score):
    if score < 5:
        return 'Low-Value'
    elif score < 9:
        return 'Mid-Value'
    else:
        return 'High-Value'

rfm['RFM_Segment_Label'] = rfm['RFM_Score'].apply(assign_segment)

rfm.head()

,Recency,Frequency,Value,R,F,M,RFM_Segment,RFM_Score,RFM_Segment_Label
Customer ID,,,,,,,,,
12346.0,326,48,-64.68,2,2,1,221,5,Mid-Value
12347.0,2,253,5633.32,4,4,4,444,12,High-Value
12348.0,75,51,2019.40,3,2,3,323,8,Mid-Value
12349.0,19,180,4404.54,4,4,4,444,12,High-Value
12350.0,310,17,334.40,2,1,2,212,5,Mid-Value


In [26]:
segment_counts = rfm['RFM_Segment_Label'].value_counts().reset_index()
segment_counts.columns = ['RFM_Segment', 'Count']
segment_counts = segment_counts.sort_values('RFM_Segment')

In [ ]:
# bar graph

fig = px.bar(segment_counts, x='RFM_Segment', y='Count', title = 'Customer Distribution by RFM Segment', labels={'RFM_Segment': 'RFM_Segment', 'Count': 'Number of Customers'}, color='RFM_Segment', color_discrete_sequence=px.colors.qualitative.Pastel)

fig.show()


In [38]:
rfm['RFM_Customer_Segments'] = ''

rfm.loc[rfm['RFM_Score'] >=9, 'RFM_Customer_Segments'] = 'Loyal'
rfm.loc[(rfm['RFM_Score']) >=6 & (rfm['RFM_Score'] <9), 'RFM_Customer_Segments'] = 'Potentially Loyal'
rfm.loc[(rfm['RFM_Score']) >=5 & (rfm['RFM_Score'] <6), 'RFM_Customer_Segments'] = 'At Risk Customers'
rfm.loc[(rfm['RFM_Score']) >=4 & (rfm['RFM_Score'] <5), 'RFM_Customer_Segments'] = "Can't Lose "
rfm.loc[(rfm['RFM_Score']) >=3 & (rfm['RFM_Score'] <4), 'RFM_Customer_Segments'] = 'Lost'

In [39]:
segment_product_counts = rfm.groupby(['RFM_Segment_Label', 'RFM_Customer_Segments']).size().reset_index(name='Count')

segment_product_counts = segment_product_counts.sort_values('Count', ascending=False)

In [41]:
#creaating TreeMap

fig_treemap_segment_product = px.treemap(segment_product_counts, path=['RFM_Segment_Label', 'RFM_Customer_Segments'], values='Count', color='RFM_Segment_Label', color_discrete_sequence=px.colors.qualitative.Pastel, title='RFM Customer Segments by Value')

fig_treemap_segment_product.show()